# Gemma 4 + Unsloth local RTX 5090 notebook

This notebook is a **local-first variant** of the stock Unsloth Gemma 4 notebook.

## What the original Unsloth notebook is
Unsloth is a high-performance training/runtime layer around the Hugging Face stack. The original notebook is a Colab demo that:
1. installs Unsloth and its dependencies,
2. loads a Gemma model in 4-bit,
3. adds LoRA adapters,
4. formats a conversational dataset,
5. runs supervised fine-tuning with TRL,
6. exports the adapters or merged model.

## What this notebook changes
- targets a **single local RTX 5090** instead of Colab,
- prefers **WSL/Linux** while still handling Windows paths,
- puts caches, datasets, checkpoints, and exports on your **SSD**,
- defaults to **Gemma 4 31B Instruct in 4-bit QLoRA**,
- expects a prepared local JSONL dataset for your Second Brain project, with a public fallback dataset for smoke tests.


## Recommended runtime
- **Best option:** Ubuntu on bare metal or **WSL2 Ubuntu**.
- **Why:** CUDA, bitsandbytes, and broader LLM tooling are usually smoother on Linux/WSL than on native Windows.
- **Windows support:** this notebook detects Windows paths, but the most reliable setup is still WSL with the SSD mounted into `/mnt/<drive>`.

Before running training, create the environment with Unsloth's current code-first install flow.

### WSL / Linux
```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
uv venv ~/.venvs/unsloth --python 3.13
source ~/.venvs/unsloth/bin/activate
uv pip install unsloth --torch-backend=auto
uv pip install datasets trl accelerate sentencepiece pillow torchvision
```

### Native Windows PowerShell
```powershell
winget install -e --id Python.Python.3.13
winget install --id=astral-sh.uv -e
uv venv $HOME\.venvs\unsloth --python 3.13
& $HOME\.venvs\unsloth\Scripts\activate
uv pip install unsloth --torch-backend=auto
uv pip install datasets trl accelerate sentencepiece pillow torchvision
```


In [ ]:
import os
import platform
from pathlib import Path

IS_WINDOWS = platform.system() == "Windows"
IS_WSL = "microsoft" in platform.release().lower() or "microsoft" in platform.version().lower()

workspace_default = Path.home() / "Jemma"
if not workspace_default.exists() and Path('/home/soumitty/Jemma').exists():
    workspace_default = Path('/home/soumitty/Jemma')

if IS_WINDOWS:
    data_root_default = Path.home() / 'JemmaData'
elif IS_WSL:
    data_root_default = Path('/mnt/d/JemmaData')
else:
    data_root_default = Path.home() / 'JemmaData'

WORKSPACE_DIR = Path(os.environ.get('JEMMA_WORKSPACE_DIR', str(workspace_default))).expanduser()
DATA_ROOT = Path(os.environ.get('JEMMA_DATA_DIR', str(data_root_default))).expanduser()
CACHE_DIR = DATA_ROOT / 'cache'
DATA_DIR = DATA_ROOT / 'datasets'
OUTPUT_DIR = DATA_ROOT / 'runs' / 'gemma4-second-brain'
EXPORT_DIR = DATA_ROOT / 'exports'
LOG_DIR = DATA_ROOT / 'logs'

for path in [DATA_ROOT, CACHE_DIR, DATA_DIR, OUTPUT_DIR, EXPORT_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(CACHE_DIR / 'hf_home')
os.environ['HF_DATASETS_CACHE'] = str(CACHE_DIR / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_DIR / 'transformers')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_DIR / 'hub')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

print({
    'is_windows': IS_WINDOWS,
    'is_wsl': IS_WSL,
    'workspace_dir': str(WORKSPACE_DIR),
    'data_root': str(DATA_ROOT),
    'cache_dir': str(CACHE_DIR),
    'output_dir': str(OUTPUT_DIR),
    'export_dir': str(EXPORT_DIR),
})
print('WSL/Linux is the recommended runtime for this notebook.')


: 

In [ ]:
import importlib.util
import sys

required_modules = ['torch', 'datasets', 'transformers', 'trl', 'unsloth']
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        'Missing packages: ' + ', '.join(missing) + '. '        'Create the environment with the install commands in the markdown cell above, then restart the kernel.'
    )

import torch
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this notebook. Start it inside your CUDA-enabled WSL/Linux or Windows environment.')

print('GPU:', torch.cuda.get_device_name(0))
print('Total VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('bfloat16 supported:', torch.cuda.is_bf16_supported())


## Model choice
- Default: `unsloth/gemma-4-31B-it`
- Why: a 5090 with 32 GB VRAM is a good fit for **4-bit QLoRA** on the 31B instruct model.
- If you want more headroom for longer context or larger effective batch size, switch to `unsloth/gemma-4-E4B-it`.


In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

MODEL_NAME = os.environ.get('JEMMA_MODEL_NAME', 'unsloth/gemma-4-31B-it')
MAX_SEQ_LENGTH = int(os.environ.get('JEMMA_MAX_SEQ_LENGTH', '4096'))
LOAD_IN_4BIT = os.environ.get('JEMMA_LOAD_IN_4BIT', '1') == '1'
FULL_FINETUNING = os.environ.get('JEMMA_FULL_FINETUNING', '0') == '1'

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=FULL_FINETUNING,
    device_map='auto',
)

tokenizer = get_chat_template(tokenizer, chat_template='gemma-4-thinking')

start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(torch.cuda.get_device_properties(0).total_memory / 1024 / 1024 / 1024, 3)
print({'model_name': MODEL_NAME, 'max_seq_length': MAX_SEQ_LENGTH, 'load_in_4bit': LOAD_IN_4BIT, 'full_finetuning': FULL_FINETUNING})
print('Reserved VRAM before LoRA (GB):', start_gpu_memory)
print('Total GPU memory (GB):', max_memory)


In [ ]:
peft_kwargs = dict(
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=int(os.environ.get('JEMMA_LORA_R', '8')),
    lora_alpha=int(os.environ.get('JEMMA_LORA_ALPHA', '16')),
    lora_dropout=0,
    bias='none',
    random_state=3407,
)

try:
    model = FastModel.get_peft_model(
        model,
        use_gradient_checkpointing='unsloth',
        **peft_kwargs,
    )
except TypeError:
    model = FastModel.get_peft_model(model, **peft_kwargs)

print('LoRA adapters attached.')


## Training data
Preferred input: a JSONL file at `DATA_DIR / "second-brain-train.jsonl"`.

Each row should contain one of these shapes:
- `{"messages": [{"role": "system|user|assistant", "content": "..."}, ...]}`
- `{"conversations": [{"role": "user|assistant", "content": "..."}, ...]}`
- `{"prompt": "...", "response": "..."}`

If that file does not exist yet, the notebook falls back to a small public dataset so you can smoke-test the pipeline immediately.


In [ ]:
from datasets import load_dataset

TRAIN_DATASET_JSONL = DATA_DIR / 'second-brain-train.jsonl'
SMOKE_TEST_ROWS = int(os.environ.get('JEMMA_SMOKE_TEST_ROWS', '3000'))

if TRAIN_DATASET_JSONL.exists():
    dataset = load_dataset('json', data_files=str(TRAIN_DATASET_JSONL), split='train')
    dataset_source = str(TRAIN_DATASET_JSONL)
else:
    dataset = load_dataset('mlabonne/FineTome-100k', split=f'train[:{SMOKE_TEST_ROWS}]')
    dataset_source = f'mlabonne/FineTome-100k[:{SMOKE_TEST_ROWS}]'

print('Loaded dataset from:', dataset_source)
print(dataset)


In [ ]:
def _normalize_turn_content(content):
    if isinstance(content, list):
        text_parts = []
        for item in content:
            if isinstance(item, dict) and item.get('type', 'text') == 'text':
                text_parts.append(item.get('text', ''))
            else:
                text_parts.append(str(item))
        return '\n'.join(part for part in text_parts if part).strip()
    return str(content).strip()


def _normalize_conversation(example):
    if example.get('conversations'):
        raw_turns = example['conversations']
    elif example.get('messages'):
        raw_turns = example['messages']
    elif 'prompt' in example and 'response' in example:
        raw_turns = [
            {'role': 'user', 'content': example['prompt']},
            {'role': 'assistant', 'content': example['response']},
        ]
    else:
        raise ValueError('Expected conversations, messages, or prompt/response fields.')

    normalized = []
    for turn in raw_turns:
        role = turn.get('role') or turn.get('from') or 'user'
        role = {'human': 'user', 'gpt': 'assistant', 'model': 'assistant'}.get(role, role)
        content = turn.get('content', turn.get('value', turn.get('text', '')))
        text = _normalize_turn_content(content)
        if not text:
            continue
        normalized.append({'role': role, 'content': [{'type': 'text', 'text': text}]})

    if len(normalized) < 2:
        raise ValueError('Each example must contain at least 2 non-empty turns.')
    return normalized


def add_conversation_field(example):
    return {'conversation': _normalize_conversation(example)}


dataset = dataset.map(add_conversation_field)

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix('<bos>')
        for convo in examples['conversation']
    ]
    return {'text': texts}


dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]['text'][:1200])


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

per_device_train_batch_size = int(os.environ.get('JEMMA_BATCH_SIZE', '1'))
gradient_accumulation_steps = int(os.environ.get('JEMMA_GRAD_ACC', '8'))
learning_rate = float(os.environ.get('JEMMA_LR', '2e-4'))
num_train_epochs = float(os.environ.get('JEMMA_EPOCHS', '1'))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field='text',
        output_dir=str(OUTPUT_DIR / 'checkpoints'),
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_steps=5,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        optim='adamw_8bit',
        weight_decay=0.001,
        lr_scheduler_type='linear',
        seed=3407,
        report_to='none',
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|turn>user\n',
    response_part='<|turn>model\n',
)

print({
    'effective_batch_size': per_device_train_batch_size * gradient_accumulation_steps,
    'learning_rate': learning_rate,
    'epochs': num_train_epochs,
})


In [ ]:
trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")


In [ ]:
messages = [{
    'role': 'user',
    'content': [{'type': 'text', 'text': 'Summarize the top three safety risks in a bridge inspection log and suggest follow-up actions.'}],
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors='pt',
    tokenize=True,
    return_dict=True,
).to('cuda')

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=False)[0])


In [ ]:
adapter_dir = EXPORT_DIR / 'gemma4-31b-second-brain-lora'
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print('Saved LoRA adapter to', adapter_dir)

SAVE_MERGED_16BIT = os.environ.get('JEMMA_SAVE_MERGED_16BIT', '0') == '1'
SAVE_GGUF = os.environ.get('JEMMA_SAVE_GGUF', '0') == '1'

if SAVE_MERGED_16BIT:
    merged_dir = EXPORT_DIR / 'gemma4-31b-second-brain-merged'
    model.save_pretrained_merged(str(merged_dir), tokenizer)
    print('Saved merged model to', merged_dir)

if SAVE_GGUF:
    gguf_dir = EXPORT_DIR / 'gemma4-31b-second-brain-gguf'
    model.save_pretrained_gguf(str(gguf_dir), tokenizer, quantization_method='Q8_0')
    print('Saved GGUF export to', gguf_dir)
